# Rabi Oscillations of a spin-1/2
This notebook simulates the time evolution of the spin under the influence of an oscillating drive field perpendicular to the quantization axis.
It covers:
- Static Spin Hamiltonian
- Rotating Wave Approximation
- Rabi Oscillations under
    - different driving strenght
    - detuning 

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from qutip import (
    tensor, qeye, sigmax, sigmay, sigmaz,
    basis, mesolve, sigmam, Qobj, Bloch
)

from spinfinity import ESR, lind, CC, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Static spin Hamiltonian
The static spin Hamiltonian includes an external field in z-direction which defines the quantization axis

In [ ]:
spin = SpinSys([0.5])
spin.gfactorvector = [2.0]
spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.0]  # T

hamil.calc_EigEnergies(spin, rebuildH=True)
f_e= float(spin.E_All_inGHz[1] - spin.E_All_inGHz[0])  # GHz

hamil.plot_ZeemanDiagramm(spin, energyscale='GHz');

## Continous-wave frequency sweep
Simulating a ESR frequency sweep also reveals the resonance frequency $f_e$. 

In [ ]:
ESR.plot_FreqSweep(spin, freq_array = np.linspace(12,16,1000), lw=0.05);

## Preparations for time evolution
The time evolution uses the Lindblad formalism, where the total spin Hamiltonian contains the static Hamiltonian, a time-dependent Hamiltonian as well as relaxation and decoherence via collapse operators. The time evolution of the density matrix is then computed from an initial state. 

### Constructing Collapse Operators
$T_1$-type operators which cause excitationand relaxation are constructed from inelastic tunneling rates. Pure adiabatic dephasing can be treated phenomenlogically via dephasing time $T_\phi$

In [ ]:
# Tunneling Parameters
spin.T = 0.2 # K
spin.V_DC = 0.0  # mV
spin.TipPolarization = [0, 0, 0.5]
spin.U = np.array([0.0])
spin.G_st = 0.1e-9  # S
spin.G_ss = [1e-9]  # S
spin.G_tt = 1 # Allow for tip-tip scattering
spin.b0 = 0.0

# Relaxation
bath.calc_Rates(spin)
bath.calc_Lifetimes(spin)
T1 = (1/spin.T_state_tot[0]+1/spin.T_state_tot[1])**-1*1e9
print(f"T1 time: {T1:.2f} ns")

c_ops_T1 = lind.calc_collapseOperators(spin, threshold_percent=1e-9, recalcRates=True)


# Decoherence
T_phi = 200  # ns
c_op_Tphi = lind.calc_dephasingOperator(spin, spin_idx=0, T_phi=T_phi)
spin.collapse_ops = c_ops_T1 + [c_op_Tphi]


### Preparing an initial state

In [ ]:
# Initial state 
# For the Rabi oscillation simulation, the system needs to be prepared in an initial state.
# Here, we prepare the system in the thermal state at the given temperature.
rho_therm = lind.initialize_thermal(spin)
bath.plot_Populations(spin, AllowPumping=False);

## Rabi Oscillation
The Rabi oscillation simulation is performed by varying the pulse duration $\tau$ and calculating the final expectation value of $\hat S_x, \hat S_y, \hat S_z$ after the pulse.

In [ ]:
# Preparing pulse parameters
# Setting the RF frequency and Rabi rate
detuning = 0.0e-3 #GHz
f_RF = f_e + detuning # GHz
RabiRate_GHz = 50*1e-3 # GHz

# Assigning the RF frequency and Rabi rate to the spin system
spin.f_RF = f_RF
spin.RabiRate = RabiRate_GHz

# Rabi Oscillation Simulation
tau_array = np.linspace(0, 200, 2000) # ns
_, result = CC.calc_RabiOscillation(spin, tau_array, e_ops=None) # if e_ops is None, the expectation values of S_x, S_y, S_z are calculated.


# Perform Oscillation Fit
x = result.times
y = result.expect[2]
x_fit, y_fit, popt, perr = hf.fit_oscillation(x, y)
A_fit, T_fit, RabiRate_fit, phi_fit, C_fit = popt


# Plot
plt.figure(figsize=(10, 5))
plt.plot(tau_array, result.expect[2], "o", ms=5, label="Simulation Data")
plt.plot(x_fit, y_fit, "-", lw=2, label=f"Fit: A={A_fit:.3f}, T={T_fit:.1f} ns, RabiRate={RabiRate_fit:.3f} GHz, phi={phi_fit:.2f}, C={C_fit:.3f}")
plt.plot()
plt.xlabel(r"Pulse Duration $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_{z} \rangle$")
plt.legend()
plt.ylim([-0.5,0.5])
plt.tight_layout()
plt.show()

# Rabi Decoherence time 
T2_Rabi = 1/(1/(2*T_phi) + 3/(4*T1))
print(f"Analytical Rabi decoherence time: {T2_Rabi:.2f} ns")

### Represenatation on the Bloch sphere

In [ ]:
sphere = Bloch()
sphere.add_points(
[
    2 * result.expect[0],
    2 * result.expect[1],
    2 * result.expect[2],
],
meth="l",
)
sphere.show()

## Rabi Oscillations for different driving strengths
A higher driving strength leads to a faster oscillation

In [ ]:
# Using different driving Rabi rates 
Rabi_rates = [10.0, 20.0, 50.0, 100.0]  # MHz
tau_array = np.linspace(0, 200, 500) # ns
y_offset = 0.7

plt.figure(figsize=(8, 10))
for i, Rabi_rate in enumerate(Rabi_rates):
    # Assigning the RF frequency and Rabi rate to the spin system
    spin.RabiRate = Rabi_rate*1e-3 # GHz

    # Rabi Oscillation Simulation
    _, result = CC.calc_RabiOscillation(spin, tau_array, e_ops=None) # if e_ops is None, the expectation values of S_x, S_y, S_z are calculated.
    xfit, yfit, _, _ = hf.fit_oscillation(result.times, result.expect[2])
    plt.plot(tau_array, result.expect[2]+y_offset*(i-1), "-", ms=5, label=f"Rabi Rate: {Rabi_rate} MHz")
    #plt.plot(xfit, yfit, "-", lw=2, label=f"Fit: Rabi Rate: {Rabi_rate} MHz")

plt.xlabel(r"Pulse Duration $\tau$ (ns)")
plt.ylabel(r"Final $\langle S_{z} \rangle$")
plt.legend()
plt.tight_layout()
plt.show()

# Restore the original Rabi rate
spin.RabiRate = RabiRate_GHz


## Detuned Rabi Oscillations
Away from resonance, the generalized Rabi frequency is

$$
f_{\mathrm{osc}}=\sqrt{f_R^2+\delta^2}.
$$

For the initial ground state, the ideal spin signal is

$$
\langle S_{z'}(t)\rangle=
\frac{\delta^2}{2(f_R^2+\delta^2)}
+\frac{f_R^2}{2(f_R^2+\delta^2)}
\cos\!\left(2\pi\sqrt{f_R^2+\delta^2}\,t\right).
$$

In [ ]:
spin.RabiRate = 20e-3 # GHz

freq_array = np.linspace(f_e-50e-3,f_e+50e-3,200)
_,_,signal_map = CC.calc_RabiDetuningMap(spin, tau_array, freq_array)

plt.figure(figsize=(8, 5))
extent = [
    tau_array[0], tau_array[-1],
    (freq_array[0]-f_e) * 1e3, (freq_array[-1]-f_e) * 1e3
]

plt.imshow(
    signal_map,
    aspect='auto',
    origin='lower',
    extent=extent,
    interpolation='none',
    cmap='magma'
)

plt.xlabel(r'Pulse Width $\tau$ (ns)')
plt.ylabel(r'Detuning $\delta = f_{\mathrm{RF}} - f_e$ (MHz)')
plt.clim([-0.5,0.5])
plt.colorbar(label=r'Final $\langle S_{z} \rangle$')
plt.tight_layout()
plt.show()